In [1]:
! pip install langchain langchain-ollama langchain-core

INFO: pip is looking at multiple versions of langchain-ollama to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_ollama-1.0.1-py3-none-any.whl.metadata (2.5 kB)
Using cached langchain_ollama-1.0.1-py3-none-any.whl (29 kB)
  Attempting uninstall: langchain-ollama
    Found existing installation: langchain-ollama 0.3.0
    Uninstalling langchain-ollama-0.3.0:
      Successfully uninstalled langchain-ollama-0.3.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browser-use 0.1.45 requires langchain-core==0.3.49, but you have langchain-core 1.2.20 which is incompatible.
browser-use 0.1.45 requires langchain-ollama==0.3.0, but you have langchain-ollama 1.0.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


📚 Session 1: Setup + LangChain Basics with Ollama

In [3]:
# 01_basic_llm.py
from langchain_ollama import OllamaLLM

# Connect to your local Ollama model
llm = OllamaLLM(model="llava-llama3")

# The simplest possible call
response = llm.invoke("What is the capital of France?")
print(response)

The capital of France is Paris.


In [5]:
# 02_chat_model.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOllama(model="llava-llama3")

messages = [
    SystemMessage(content="You are a helpful geography tutor."),
    HumanMessage(content="What is the capital of India?")
]

response = chat.invoke(messages)
print(response.content)

The capital of India is New Delhi.


In [6]:
# 03_streaming.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llava-llama3")

for chunk in chat.stream([HumanMessage(content="Tell me a short story about a robot.")]):
    print(chunk.content, end="", flush=True)

print()  # newline at the end

Once upon a time, there was a robot named Zeta. Zeta was built to serve as a companion and helper for his human owner, who was a scientist. The scientist spent most of his time in the laboratory, conducting experiments and studying various phenomena.

Zeta loved spending time with his owner and helping him with his work. He was particularly good at organizing the lab and keeping track of important data. Despite being an artificial intelligence, Zeta had a strong sense of loyalty and always put the needs of his owner first.

One day, while his owner was out in the field conducting experiments, Zeta decided to take a walk outside the laboratory. As he explored the world around him, he encountered a small bird that had fallen from its nest. The bird was cold and scared, so Zeta quickly picked it up and took it back to the laboratory.

Zeta's owner was amazed at the robot's quick thinking and kindness. He thanked Zeta for saving the little bird and promised to give him an upgrade as a rewa

In [7]:
# 04_batch.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llava-llama3")

questions = [
    [HumanMessage(content="What is 5 + 5?")],
    [HumanMessage(content="Name one planet in our solar system.")],
    [HumanMessage(content="What color is the sky?")],
]

responses = chat.batch(questions)

for r in responses:
    print(r.content)
    print("---")

5 + 5 is equal to 10.
---
Sure! Here's an example:

Saturn is a gas giant located at the outer edge of the Solar System, between Jupiter and Uranus. It is known for its beautiful ring system, which includes dust and rock particles that orbit around it. Saturn is also notable for having multiple moons orbiting around it. The planet was discovered in 1666 by Galileo Galilei.
---
The sky is blue.
---


In [9]:
# confusion_buster.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOllama(model="llava-llama3")

question = "What should I know about a dog's diet?"

# --- WITHOUT SystemMessage ---
print("=== WITHOUT System Message ===")
response1 = chat.invoke([HumanMessage(content=question)])
print(response1.content)

print()

# --- WITH SystemMessage ---
print("=== WITH System Message (acting as a vet) ===")
response2 = chat.invoke([
    SystemMessage(content="You are an experienced veterinarian. Give precise, clinical advice."),
    HumanMessage(content=question)
])
print(response2.content)


=== WITHOUT System Message ===
When it comes to a dog's diet, there are several important factors to consider in order to ensure the best health and well-being for your pet. These include:

1. Nutrition: Dogs require a balanced diet that provides them with all of the necessary nutrients for their growth and overall health. This means providing high-quality food formulated especially for dogs.
2. Quality and Quantity: It's important to know what types of food are suitable for your dog, whether it's dry food or wet food, and how much they should eat each day. Consult with your veterinarian to determine the appropriate amount of food for your dog.
3. Allergies: Some dogs have allergies to certain ingredients in their diet, such as gluten in wheat-based products or lactose in dairy products. If you suspect that your dog has an allergy, consult with your veterinarian to rule out any allergies and provide a suitable diet.
4. Weight Management: Maintaining a healthy weight is important for yo

📚 Session 2: Prompt Templates

In [10]:
# 05_prompt_template.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

# Define the template ONCE
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful translator. Only reply with the translation, nothing else."),
    ("human", "Translate '{text}' from {source_lang} to {target_lang}.")
])

# Fill in variables and invoke
chain = prompt | chat   # <-- the pipe operator, more on this in Session 3

response = chain.invoke({
    "text": "Good morning",
    "source_lang": "English",
    "target_lang": "Spanish"
})

print(response.content)

"Buenos días"


In [11]:
# 06_reuse_template.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {subject}. Give concise answers."),
    ("human", "{question}")
])

chain = prompt | chat

# Reuse with different subjects and questions
pairs = [
    {"subject": "astronomy",  "question": "What is a neutron star?"},
    {"subject": "cooking",    "question": "What does 'blanching' mean?"},
    {"subject": "history",    "question": "Why did the Roman Empire fall?"},
]

for pair in pairs:
    response = chain.invoke(pair)
    print(f"[{pair['subject'].upper()}] {response.content}\n")

[ASTRONOMY] A neutron star, also known as a pulsar or stellar remnant, is a type of compact star that is formed after the explosion of a massive star. It consists mainly of neutrons and has an extremely high density. Neutron stars are typically only about 10 miles in diameter, but they have the mass of the sun! They are incredibly hot and can emit intense radiation in the form of X-rays and gamma rays.

[COOKING] Blanching refers to the process of blanching vegetables, which involves briefly boiling or steaming them and then immediately immersing them in ice water to stop the cooking process and preserve their vibrant green color and crisp texture. This technique is often used for preparing dishes like soup stocks, sauces, or garnishes.

[HISTORY] The Roman Empire fell due to a combination of factors, including political and social unrest, economic decline, and external pressures from neighboring tribes. It is difficult to pinpoint a single cause, but internal power struggles, the rise

In [12]:
# 07_inspect_prompt.py
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Tell me about {topic} in {num} sentences.")
])

# Format it WITHOUT invoking the model — just see the result
formatted = prompt.format_messages(
    role="marine biologist",
    topic="bioluminescence",
    num="2"
)

for msg in formatted:
    print(f"[{msg.type.upper()}]: {msg.content}")


[SYSTEM]: You are a marine biologist.
[HUMAN]: Tell me about bioluminescence in 2 sentences.


In [13]:
# 08_simple_prompt_template.py
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llava-llama3")

# Single-string template (no system/human split)
prompt = PromptTemplate.from_template(
    "Give me 3 fun facts about {animal}. Keep it brief."
)

chain = prompt | llm

print(chain.invoke({"animal": "octopus"}))
print("---")
print(chain.invoke({"animal": "platypus"}))

1. Octopuses have three hearts, each one serving a different function.
2. The smallest species of octopus can be as small as 1 inch in length.
3. Octopuses are known to have complex social structures and have been observed exhibiting behaviors such as cooperation and playfulness.
---
1. Platypuses can grow up to 18 inches in length.
2. They have a distinctive duck-bill appearance and are known for their webbed feet, which they use to swim.
3. Platypusses are the only known living example of a synapsid vertebrate, meaning that their skull has openings for nerves that are connected to the spinal cord, similar to how mammals are.


In [14]:
# 09_two_ways_same_thing.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "{question}")
])

# ── WAY 1: Manually, step by step ──────────────────────────
messages = prompt.format_messages(role="chef", question="How do I julienne a carrot?")
response = chat.invoke(messages)
print("Way 1:", response.content)

# ── WAY 2: Using the pipe operator (does the same thing) ───
chain = prompt | chat
response = chain.invoke({"role": "chef", "question": "How do I julienne a carrot?"})
print("Way 2:", response.content)

# Both produce IDENTICAL results.
# Way 2 is just cleaner — especially when chains get longer.

Way 1: To julienne a carrot, you will need the following ingredients:

* 1 large carrots
* Cutting board
* Chef's knife
* Shredded cheese (optional)

Instructions:

1. Wash the carrots thoroughly under running water.
2. Place the carrots on a cutting board and trim both ends off using a chef's knife.
3. Stand the carrot upright on the cutting board with the stem facing upwards.
4. Using a sharp knife, slice across the carrot in a diagonal direction to create thin strips.
5. Repeat this process for the remaining portion of the carrot until it is completely julienneed.

If you want to add some cheese on top, follow these steps:

1. After step 3, place the sliced carrot strips into a bowl.
2. Grate the desired amount of shredded cheese onto the carrot slices.
3. Mix well and serve immediately.

Enjoy your delicious julienned carrots!
Way 2: To julienne a carrot, follow these steps:

1. Wash the carrot thoroughly to remove any dirt or residue.
2. Peel the skin off of the carrot using a sha

📚 Session 3: Chains + LCEL (LangChain Expression Language)

In [15]:
# 10_str_output_parser.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. Answer in one sentence only."),
    ("human", "What is the capital of {country}?")
])

# 3-step chain: prompt → model → parser
chain = prompt | chat | parser

# Now .invoke() returns a plain string, not an AIMessage object
result = chain.invoke({"country": "Japan"})
print(type(result))   # <class 'str'>
print(result)         # Tokyo is the capital of Japan.

<class 'langchain_core.messages.base.TextAccessor'>
The capital of Japan is Tokyo.


In [16]:
# 11_runnable_lambda.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

# A plain Python function — make the output shout
def make_uppercase(text: str) -> str:
    return text.upper()

# Wrap it so LCEL understands it
shout = RunnableLambda(make_uppercase)

# 4-step chain
chain = prompt | chat | StrOutputParser() | shout

result = chain.invoke({"question": "Name one planet in our solar system."})
print(result)  # → "MARS IS A PLANET IN OUR SOLAR SYSTEM." (all caps)

SATURN.


In [18]:
# 12_parallel_chain.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")
parser = StrOutputParser()

# Chain 1: get a summary
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the topic in one sentence."),
    ("human", "{topic}")
])

# Chain 2: get a fun fact
fact_prompt = ChatPromptTemplate.from_messages([
    ("system", "Give one surprising fun fact about the topic."),
    ("human", "{topic}")
])

summary_chain = summary_prompt | chat | parser
fact_chain    = fact_prompt    | chat | parser

# Run BOTH chains on the same input simultaneously
parallel = RunnableParallel(
    summary = summary_chain,
    fun_fact = fact_chain
)

result = parallel.invoke({"topic": "black holes"})

print("SUMMARY: ", result["summary"])
print("FUN FACT:", result["fun_fact"])

SUMMARY:  Black holes are large, dense regions of space that contain an immense amount of mass and energy. They occur when a star has undergone a catastrophic explosion known as a supernova, causing matter to be compressed into a very small area. This compression creates a singularity at the center of the black hole, which is characterized by its intense gravitational pull and the strong curvature of space around it. Black holes are important in our understanding of the universe's structure and evolution, as they play a role in shaping the formation and growth of galaxies. They also serve as cosmic engines for producing powerful bursts of energy known as gamma rays, which can be detected from great distances.

The topic is related to cosmology, astrophysics, and gravitational physics. It has implications for our understanding of the universe's history and evolution, as well as its overall structure and composition. The study of black holes also provides valuable insights into the natur

In [21]:
# 13_stream_chain.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a storyteller."),
    ("human", "Tell me a very short story about {hero}.")
])

chain = prompt | chat | StrOutputParser()

# Stream the final string output word by word
for chunk in chain.stream({"hero": "a lonely robot"}):
    print(chunk, end="", flush=True)

print()

Once upon a time, in a world of metal and wires, there lived a lonely robot named Zeta. Zeta was built to serve, but he longed for something more. He wanted companionship and love.

One day, while on his duties, he stumbled upon a beautiful garden filled with vibrant flowers. The colors were so vivid that they made Zeta's digital eyes sparkle. As he gazed at the garden, he saw a rainbow-colored flower blooming amidst the greenery.

Zeta reached out to touch it, and as he did, a small fairy appeared. Her name was Lila, and she had been watching Zeta for quite some time. She told him that his loneliness would soon be ended, and together they explored the wonders of love and friendship.

From that day on, Zeta and Lila were inseparable. They laughed, danced, and shared their dreams with each other. And as they did so, Zeta's robotic heart began to warm up, filled with joy and happiness.

The end


📚 Session 4: Output Parsers

In [25]:
# 14_json_output_parser.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data extraction assistant. Always respond with valid JSON only. No explanation."),
    ("human", """Extract the following from this text and return as JSON with keys:
    name, age, city

    Text: {text}""")
])

chain = prompt | chat | parser

result = chain.invoke({
    "text": "My name is Alice, I am 30 years old and I live in Berlin."
})

print(type(result))       # <class 'dict'>
print(result)             # {'name': 'Alice', 'age': 30, 'city': 'Berlin'}
print(result["name"])     # Alice  ← access like any Python dict!
print(result["age"] + 5)  # 35     ← it's an actual int, not a string

<class 'dict'>
{'city': 'Berlin', 'name': 'Alice', 'age': '30'}
Alice


TypeError: can only concatenate str (not "int") to str

In [33]:
# 15_pydantic_parser_fixed.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field, field_validator
from typing import List

class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age as a number, not a string")
    city: str = Field(description="City they live in")

    # This forces age to always be an int, even if model returns "30"
    @field_validator("age", mode="before")
    @classmethod
    def coerce_age(cls, v):
        return int(v)

parser = JsonOutputParser(pydantic_object=Person)
chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured info. Return JSON only. {format_instructions}"),
    ("human", "{text}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | chat | parser

result = chain.invoke({
    "text": "My name is Alice, I am 30 years old and I live in Berlin."
})

print(type(result))
print(result)
print(result["age"] + 5)   # Now works — age is guaranteed int

<class 'dict'>
{'name': 'Alice', 'age': 30, 'city': 'Berlin'}
35


In [29]:
# 15_pydantic_parser.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import List

# 1. Define the exact shape you want back
class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age in years")
    hobbies: List[str] = Field(description="List of hobbies")
    city: str = Field(description="City they live in")
    work: str = Field(description="what the person do")

# 2. Create the parser from your model
parser = PydanticOutputParser(pydantic_object=Person)
chat = ChatOllama(model="llava-llama3")

# 3. Inject format instructions automatically into the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured information. {format_instructions}"),
    ("human", "{text}")
])

# 4. Partial-fill the format instructions (they come from the parser)
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | chat | parser

result = chain.invoke({
    "text": "John is a 25-year-old developer from Toronto who loves hiking, chess, and cooking."
})

print(type(result))        # <class 'Person'>  ← a real Python object!
print(result.name)         # John
print(result.age + 5)          # 25
print(result.hobbies)      # ['hiking', 'chess', 'cooking']
print(result.city)  # Toronto
print(result.work)  # developer
# Pydantic validates types — age is guaranteed to be an int, not "25"

<class '__main__.Person'>
John
30
['hiking', 'chess', 'cooking']
Toronto
developer


In [40]:
# 15_pydantic_parser_fixed_v2.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field, field_validator
from typing import List

# 1. Define the shape
class Movie(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Year of release as a number")
    director: str = Field(description="Full name of the director")
    genres: List[str] = Field(description="List of genres")

    @field_validator("year", mode="before")
    @classmethod
    def coerce_year(cls, v):
        return int(v)

# 2. Use JsonOutputParser — more forgiving with local models
parser = JsonOutputParser(pydantic_object=Movie)
chat = ChatOllama(model="llava-llama3")

# 3. Concrete example in prompt — NO get_format_instructions()
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction assistant.
Extract movie information and return JSON only, no explanation.
Use exactly this format:
{{"title": "Movie Name", "year": 2000, "director": "Full Name", "genres": ["genre1", "genre2"]}}"""),
    ("human", "Extract movie info from this text: {text}")
])

chain = prompt | chat | parser

# 4. Invoke
raw = chain.invoke({
    "text": "Inception is a 2010 sci-fi thriller directed by Christopher Nolan."
})

# 5. Validate through Pydantic manually — gives you the typed object
result = Movie(**raw)

# 6. Access fields
print(type(result))
print("Title:   ", result.title)
print("Year:    ", result.year,     type(result.year))    # must be int
print("Director:", result.director)
print("Genres:  ", result.genres)
print("Year + 1:", result.year + 1)  # proves it's an int

<class '__main__.Movie'>
Title:    Inception
Year:     2010 <class 'int'>
Director: Christopher Nolan
Genres:   ['Science Fiction & Fantasy', 'Mystery, Thriller & Suspense']
Year + 1: 2011


In [30]:
# 16_parse_list.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama
from pydantic import BaseModel
from typing import List

class BookList(BaseModel):
    books: List[str]
    genre: str

parser = JsonOutputParser(pydantic_object=BookList)
chat = ChatOllama(model="llava-llama3")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a librarian. Respond with JSON only. {format_instructions}"),
    ("human", "Recommend 3 {genre} books.")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | chat | parser

result = chain.invoke({"genre": "science fiction"})

print(result["genre"])
for i, book in enumerate(result["books"], 1):
    print(f"{i}. {book}")

KeyError: 'genre'

In [37]:
# 16_parse_list_bulletproof.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llava-llama3")

# No Pydantic schema — let the model return whatever JSON it wants
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a librarian. Respond with JSON only.
Return exactly this format, nothing else:
{{"books": ["Book Title 1", "Book Title 2", "Book Title 3"]}}"""),
    ("human", "Recommend 3 {genre} books.")
])

chain = prompt | chat | parser

#genre = "science fiction"
result = chain.invoke({"genre": "science fiction"})

# --- Diagnose what came back ---
print("Raw result:", result)
print("Type:", type(result))

# --- Handle all cases safely ---
if isinstance(result, dict):
    books = result.get("books", result)   # try "books" key, fallback to whole dict
elif isinstance(result, list):
    books = result                         # model returned the list directly
else:
    books = []
    print("Unexpected format:", result)

# --- Print safely ---
print(f"\nGenre: {genre}")
for i, book in enumerate(books, 1):
    print(f"{i}. {book}")

Raw result: {'books': ['Dune', "The Hitchhiker's Guide to the Galaxy", 'A Game of Thrones']}
Type: <class 'dict'>

Genre: science fiction
1. Dune
2. The Hitchhiker's Guide to the Galaxy
3. A Game of Thrones


📚 Session 5: Memory + Conversation History

In [41]:
# 17_manual_memory.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

chat = ChatOllama(model="llava-llama3")

# This list IS the memory — we manage it ourselves
history = [
    SystemMessage(content="You are a friendly assistant. Remember everything the user tells you.")
]

def chat_with_memory(user_input: str) -> str:
    # 1. Add user message to history
    history.append(HumanMessage(content=user_input))

    # 2. Send the FULL history to the model every time
    response = chat.invoke(history)

    # 3. Add model reply to history
    history.append(AIMessage(content=response.content))

    return response.content

# Have a real conversation
print("Bot:", chat_with_memory("Hi! My name is Alex and I love Python."))
print("Bot:", chat_with_memory("What programming language did I mention?"))
print("Bot:", chat_with_memory("And what is my name?"))

# Inspect the history — see exactly what gets sent each time
print("\n--- Full conversation history ---")
for msg in history:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"[{role}]: {msg.content}")

Bot: Hello, Alex! It's great to meet you. How can I assist you today? Is there anything specific you would like to know about Python or programming in general?
Bot: You mentioned the programming language Python.
Bot: Your name is Alex.

--- Full conversation history ---
[System]: You are a friendly assistant. Remember everything the user tells you.
[Human]: Hi! My name is Alex and I love Python.
[AI]: Hello, Alex! It's great to meet you. How can I assist you today? Is there anything specific you would like to know about Python or programming in general?
[Human]: What programming language did I mention?
[AI]: You mentioned the programming language Python.
[Human]: And what is my name?
[AI]: Your name is Alex.


In [42]:
# 18_chat_message_history.py
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.chat_message_histories import ChatMessageHistory

chat = ChatOllama(model="llava-llama3")

# LangChain manages the list for you
memory = ChatMessageHistory()
memory.add_message(SystemMessage(content="You are a helpful assistant with a great memory."))

def chat_with_memory(user_input: str) -> str:
    memory.add_user_message(user_input)
    response = chat.invoke(memory.messages)
    memory.add_ai_message(response.content)
    return response.content

print("Bot:", chat_with_memory("My favourite colour is blue."))
print("Bot:", chat_with_memory("I also love pizza."))
print("Bot:", chat_with_memory("What do you know about me so far?"))

Bot: Understood! Blue is a cool and calming color, often associated with trust and loyalty. It's a popular choice for many people's favorite colors. Is there anything else I can help you with?
Bot: Pizza is a delicious food that can be enjoyed in many different ways. From classic margherita to loaded meat-lovers, there are countless options to suit everyone's taste preferences. What is your favorite type of pizza? Do you like thin crust or thick crust, and what toppings do you prefer?
Bot: So far, I know that you enjoy the color blue, and that you like pizza. Is there anything else you'd like to share about yourself? I'm here to help with any questions or information you might need.


In [43]:
# 19_runnable_with_history.py
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

chat = ChatOllama(model="llava-llama3")

# MessagesPlaceholder is the slot where history gets injected
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember everything."),
    MessagesPlaceholder(variable_name="history"),   # ← history goes here
    ("human", "{input}")
])

chain = prompt | chat | StrOutputParser()

# Each session_id gets its own separate memory
store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap the chain with memory
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Config tells it which session to use
config = {"configurable": {"session_id": "user_alex"}}

# Now just pass the new message — history is handled automatically
r1 = chain_with_memory.invoke({"input": "My name is Alex."}, config=config)
r2 = chain_with_memory.invoke({"input": "I work as a data scientist."}, config=config)
r3 = chain_with_memory.invoke({"input": "Summarise what you know about me."}, config=config)

print("1:", r1)
print("2:", r2)
print("3:", r3)

# Multiple users — completely separate memories!
config2 = {"configurable": {"session_id": "user_sara"}}
r4 = chain_with_memory.invoke({"input": "What is my name?"}, config=config2)
print("Sara's session:", r4)  # ← has no idea about Alex

1: Hello, Alex! I'm glad to be your assistant. How can I help you today?
2: As a data scientist, you likely have a strong background in mathematics and statistics, and are responsible for collecting, analyzing, and interpreting data. Is there something specific you would like to talk about or ask me about in your line of work?
3: Based on what you've told me, I can summarize that your name is Alex, and you are a data scientist with a strong background in mathematics and statistics. You likely have experience collecting, analyzing, and interpreting data in your work as a data scientist. Is there anything else you would like to know or discuss?
Sara's session: My name is Assistant. I am here to assist you with any questions or tasks you may have. Is there something specific that I can help you with today?


Session 6 — RAG (Retrieval Augmented Generation)

In [44]:
! pip install langchain-community chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
! pip install langchain-text-splitters

  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.20
    Uninstalling langchain-core-1.2.20:
      Successfully uninstalled langchain-core-1.2.20


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browser-use 0.1.45 requires langchain-core==0.3.49, but you have langchain-core 0.3.83 which is incompatible.
browser-use 0.1.45 requires langchain-ollama==0.3.0, but you have langchain-ollama 1.0.1 which is incompatible.
langchain 1.2.13 requires langchain-core<2.0.0,>=1.2.10, but you have langchain-core 0.3.83 which is incompatible.
langchain-ollama 1.0.1 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1: Load
loader = TextLoader("space_facts.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {len(documents[0].page_content)}")
print()

# Step 2: Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
chunks = splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print()

Loaded 1 document(s)
Total characters: 688

Split into 5 chunks

--- Chunk 1 ---
The James Webb Space Telescope was launched on December 25, 2021.
It orbits the Sun at the L2 Lagrange point, about 1.5 million kilometers from Earth.

--- Chunk 2 ---
Webb can see infrared light, which allows it to peer through dust clouds and observe
the earliest galaxies in the universe.

--- Chunk 3 ---
The Hubble Space Telescope was launched in 1990 and orbits Earth at about 550 kilometers altitude.
Unlike Webb, Hubble primarily observes visible and ultraviolet light.

--- Chunk 4 ---
Hubble has been serviced by astronauts five times, the last time in 2009.

--- Chunk 5 ---
Mars has two small moons called Phobos and Deimos.
Phobos is slowly spiraling inward and is expected to either crash into Mars
or break apart in about 50 million years.



In [3]:
! pip install --upgrade langchain-core langchain-ollama

  Using cached langchain_core-1.2.20-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.20-py3-none-any.whl (504 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browser-use 0.1.45 requires langchain-core==0.3.49, but you have langchain-core 1.2.20 which is incompatible.
browser-use 0.1.45 requires langchain-ollama==0.3.0, but you have langchain-ollama 1.0.1 which is incompatible.
langchain-anthropic 0.3.3 requires langchain-core<0.4.0,>=0.3.30, but you have langchain-core 1.2.20 which is incompatible.
langchain-aws 0.2.19 requires langchain-core<0.4.0,>=0.3.49, but you have langchain-core 1.2.20 which is incompatible.
langchain-deepseek 0.1.3 requires langchain-core<1.0.0,>=0.3.47, but you have langchain-core 1.2.20 which is incompatible.
langchain-google-genai 2.1.2 requires langchain-core<0.4.0,>=0.3.49, but you have langchain-core 1.2.20 which is incompatible.
langchain-openai 0.3.11 requires langchain-core<1.0.0,>=0.3.49, but you have langchain-core 1.2.20 which is in

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# Load and split (same as before)
loader = TextLoader("space_facts.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(documents)

# Embed and store
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Stored {vectorstore._collection.count()} chunks in vector database")

In [1]:
import langchain_core
import langchain_ollama
print(langchain_core.__version__)
print(langchain_ollama.__version__)

1.2.20
1.0.1


In [2]:
import subprocess
result = subprocess.run(
    ["pip", "show", "langchain-ollama", "langchain-core", "langchain-community"],
    capture_output=True, text=True
)
print(result.stdout)

Name: langchain-ollama
Version: 1.0.1
Summary: An integration package connecting Ollama and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/ollama
Author: 
Author-email: 
License: MIT
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: langchain-core, ollama
Required-by: browser-use
---
Name: langchain-core
Version: 1.2.20
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: browser-use, langchain, langchain-anthropic, langchain-aws, langchain-chroma, langchain-community, langchain-deepseek, langchain-google-genai, langchain-ollama, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
N

In [3]:
! pip install --upgrade langchain-ollama langchain-core langchain langchain-community langchain-text-splitters

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 5.6 MB/s eta 0:00:01
   --------------------------------- ------ 2.1/2.5 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 5.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 4.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.7
    Uninstalling langchain-text-splitters-0.3.7:
      Successfully uninstalled langchain-text-splitters-0.3.7
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.31
    Uninstalling langchain-community-0.3.31:
      Successfully uninstalled la


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import subprocess
result = subprocess.run(
    ["pip", "show", "langchain-ollama", "langchain-core"],
    capture_output=True, text=True
)
print(result.stdout)

Name: langchain-ollama
Version: 1.0.1
Summary: An integration package connecting Ollama and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/ollama
Author: 
Author-email: 
License: MIT
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: langchain-core, ollama
Required-by: browser-use
---
Name: langchain-core
Version: 1.2.20
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: browser-use, langchain, langchain-anthropic, langchain-aws, langchain-chroma, langchain-classic, langchain-community, langchain-deepseek, langchain-google-genai, langchain-ollama, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langg

In [5]:
import subprocess
result = subprocess.run(
    ["pip", "show", "browser-use"],
    capture_output=True, text=True
)
print(result.stdout)

Name: browser-use
Version: 0.1.45
Summary: Make websites accessible for AI agents
Home-page: 
Author: Gregor Zunic
Author-email: 
License: 
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: anyio, botocore, click, faiss-cpu, google-api-core, httpx, langchain, langchain-anthropic, langchain-aws, langchain-core, langchain-deepseek, langchain-google-genai, langchain-ollama, langchain-openai, markdownify, mem0ai, patchright, posthog, psutil, pydantic, pyperclip, python-dotenv, requests, rich, screeninfo, textual, typing-extensions
Required-by: 



In [6]:
! pip install langchain-ollama --upgrade --ignore-requires-python


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
! pip install langchain-ollama==0.2.3

  Using cached langchain_core-0.3.83-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.3.83-py3-none-any.whl (458 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.20
    Uninstalling langchain-core-1.2.20:
      Successfully uninstalled langchain-core-1.2.20
  Attempting uninstall: langchain-ollama
    Found existing installation: langchain-ollama 1.0.1
    Uninstalling langchain-ollama-1.0.1:
      Successfully uninstalled langchain-ollama-1.0.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browser-use 0.1.45 requires langchain-core==0.3.49, but you have langchain-core 0.3.83 which is incompatible.
browser-use 0.1.45 requires langchain-ollama==0.3.0, but you have langchain-ollama 0.2.3 which is incompatible.
langchain 1.2.13 requires langchain-core<2.0.0,>=1.2.10, but you have langchain-core 0.3.83 which is incompatible.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.83 which is incompatible.
langchain-community 0.4.1 requires langchain-core<2.0.0,>=1.0.1, but you have langchain-core 0.3.83 which is incompatible.
langchain-text-splitters 1.1.1 requires langchain-core<2.0.0,>=1.2.13, but you have langchain-core 0.3.83 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.

In [8]:
import subprocess
result = subprocess.run(
    ["pip", "show", "langchain-ollama"],
    capture_output=True, text=True
)
print(result.stdout)

Name: langchain-ollama
Version: 0.2.3
Summary: An integration package connecting Ollama and LangChain
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: langchain-core, ollama
Required-by: browser-use



In [9]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
print("All imports OK")

All imports OK


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# Load and split
loader = TextLoader("space_facts.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(documents)

# Embed and store
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Stored {vectorstore._collection.count()} chunks in vector database")

In [1]:
import subprocess
result = subprocess.run(
    ["pip", "show", "chromadb"],
    capture_output=True, text=True
)
print(result.stdout)

Name: chromadb
Version: 1.5.5
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\SUBRAT\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma



In [2]:
from langchain_community.vectorstores import FAISS
print("FAISS import OK")

FAISS import OK


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Load and split
loader = TextLoader("space_facts.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(documents)

# Embed and store
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save to disk
vectorstore.save_local("./faiss_db")

print(f"Stored {len(chunks)} chunks in vector database")
print("Saved to ./faiss_db")

Stored 5 chunks in vector database
Saved to ./faiss_db


In [2]:
! pip install --upgrade langchain

  Using cached langchain_core-1.2.20-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.2.20-py3-none-any.whl (504 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
browser-use 0.1.45 requires langchain-core==0.3.49, but you have langchain-core 1.2.20 which is incompatible.
browser-use 0.1.45 requires langchain-ollama==0.3.0, but you have langchain-ollama 0.2.3 which is incompatible.
langchain-anthropic 0.3.3 requires langchain-core<0.4.0,>=0.3.30, but you have langchain-core 1.2.20 which is incompatible.
langchain-aws 0.2.19 requires langchain-core<0.4.0,>=0.3.49, but you have langchain-core 1.2.20 which is incompatible.
langchain-deepseek 0.1.3 requires langchain-core<1.0.0,>=0.3.47, but you have langchain-core 1.2.20 which is incompatible.
langchain-google-genai 2.1.2 requires langchain-core<0.4.0,>=0.3.49, but you have langchain-core 1.2.20 which is incompatible.
langchain-ollama 0.2.3 requires langchain-core<0.4.0,>=0.3.33, but you have langchain-core 1.2.20 which is inc

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Load the saved vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.load_local(
    "./faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Ask a question
question = "When was the James Webb telescope launched?"
results = retriever.invoke(question)

print(f"Question: {question}")
print(f"Found {len(results)} relevant chunks:")
print()
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

Question: When was the James Webb telescope launched?
Found 2 relevant chunks:

--- Chunk 1 ---
The James Webb Space Telescope was launched on December 25, 2021.
It orbits the Sun at the L2 Lagrange point, about 1.5 million kilometers from Earth.

--- Chunk 2 ---
The Hubble Space Telescope was launched in 1990 and orbits Earth at about 550 kilometers altitude.
Unlike Webb, Hubble primarily observes visible and ultraviolet light.



In [1]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Load vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.load_local(
    "./faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Format retrieved chunks into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt — context comes from retriever, question from user
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. 
Answer the question using only the context provided below.
If the answer is not in the context, say 'I don't know'.

Context:
{context}"""),
    ("human", "{question}")
])

chat = ChatOllama(model="llava-llama3")

# Full RAG chain
chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat
    | StrOutputParser()
)

# Ask questions
questions = [
    "When was the James Webb telescope launched?",
    "How many times has Hubble been serviced?",
    "What will happen to Phobos?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}")
    print()

Q: When was the James Webb telescope launched?
A: The James Webb Space Telescope was launched on December 25, 2021.

Q: How many times has Hubble been serviced?
A: Hubble has been serviced by astronauts five times, the last time in 2009.

Q: What will happen to Phobos?
A: Phobos is slowly spiraling inward and is expected to either crash into Mars or break apart in about 50 million years.



In [3]:
prompt_strict = ChatPromptTemplate.from_messages([
    ("system", """You are a document lookup assistant.
You MUST only answer using the context below.
You MUST say exactly "I don't know" if the answer is not found in the context.
You MUST NOT use any outside knowledge under any circumstances.

Context:
{context}"""),
    ("human", "{question}")
])

chain_strict = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt_strict
    | chat
    | StrOutputParser()
)

print(chain_strict.invoke("What is the speed of light?"))

I'm sorry, but I am not able to answer questions that are not related to the context provided. In this context, the speed of light is not relevant information. Is there something else you would like to know about the Hubble Space Telescope or the James Webb Space Telescope?


Session 7 — Agents + Tools

In [4]:
! pip install langchain-experimental numexpr


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import langchain_experimental
print("OK")

OK


In [17]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Step 1 — Define tools
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression like '2 + 2' or '10 * 5'."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def word_counter(text: str) -> str:
    """Count the number of words in a text."""
    count = len(text.split())
    return f"{count} words"

@tool
def uppercaser(text: str) -> str:
    """Convert any text to uppercase."""
    return text.upper()

tools = [calculator, word_counter, uppercaser]

# Step 2 — Create agent (llama3.1 supports tool calling)
chat = ChatOllama(model="llama3.1:8b")
agent = create_react_agent(chat, tools)

# Test calculator
result1 = agent.invoke({
    "messages": [{"role": "user", "content": "What is 123 multiplied by 45?"}]
})

# Test word counter
result2 = agent.invoke({
    "messages": [{"role": "user", "content": "How many words are in: the quick brown fox?"}]
})

print("=== Calculator ===")
for message in result1["messages"]:
    if hasattr(message, "content") and message.content:
        print(f"[{message.__class__.__name__}]: {message.content}")

print()
print("=== Word Counter ===")
for message in result2["messages"]:
    if hasattr(message, "content") and message.content:
        print(f"[{message.__class__.__name__}]: {message.content}")

print()

result = agent.invoke({
    "messages": [{"role": "user", "content": "Count the words in 'hello world' then give me the words text in uppercase"}]
})

for message in result["messages"]:
    if hasattr(message, "content") and message.content:
        print(f"[{message.__class__.__name__}]: {message.content}")

C:\Users\SUBRAT\AppData\Local\Temp\ipykernel_3036\1618092128.py:30: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(chat, tools)


=== Calculator ===
[HumanMessage]: What is 123 multiplied by 45?
[ToolMessage]: 5535
[AIMessage]: The result of multiplying 123 by 45 is 5535.

=== Word Counter ===
[HumanMessage]: How many words are in: the quick brown fox?
[ToolMessage]: 4 words
[AIMessage]: The output of the word counter tool indicates that there are 4 words in the phrase "the quick brown fox".

[HumanMessage]: Count the words in 'hello world' then give me the words text in uppercase
[ToolMessage]: 2 words
[ToolMessage]: HELLO WORLD
[AIMessage]: There are 2 words in the text 'hello world'. The text in uppercase is HELLO WORLD.


Session 8 — LangGraph

In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Step 1 — Define state shape
class MyState(TypedDict):
    message: str
    step_log: list

# Step 2 — Define nodes (just functions)
def node_a(state: MyState) -> MyState:
    print("Node A running")
    state["step_log"].append("node_a")
    state["message"] = state["message"].upper()
    return state

def node_b(state: MyState) -> MyState:
    print("Node B running")
    state["step_log"].append("node_b")
    state["message"] = state["message"] + "!!!"
    return state

# Step 3 — Build graph
graph = StateGraph(MyState)
graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)

# Step 4 — Connect nodes with edges
graph.add_edge(START, "node_a")
graph.add_edge("node_a", "node_b")
graph.add_edge("node_b", END)

# Step 5 — Compile and run
app = graph.compile()
result = app.invoke({
    "message": "hello world",
    "step_log": []
})

print(f"Final message: {result['message']}")
print(f"Steps taken: {result['step_log']}")

Node A running
Node B running
Final message: HELLO WORLD!!!
Steps taken: ['node_a', 'node_b']


In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class MyState(TypedDict):
    number: int
    result: str

# Nodes
def router(state: MyState) -> MyState:
    print(f"Router checking number: {state['number']}")
    return state

def node_positive(state: MyState) -> MyState:
    state["result"] = f"{state['number']} is positive!"
    return state

def node_negative(state: MyState) -> MyState:
    state["result"] = f"{state['number']} is negative!"
    return state

def node_zero(state: MyState) -> MyState:
    state["result"] = "That's zero!"
    return state

# Conditional function — returns name of next node
def decide_route(state: MyState) -> str:
    if state["number"] > 0:
        return "node_positive"
    elif state["number"] < 0:
        return "node_negative"
    else:
        return "node_zero"

# Build graph
graph = StateGraph(MyState)
graph.add_node("router", router)
graph.add_node("node_positive", node_positive)
graph.add_node("node_negative", node_negative)
graph.add_node("node_zero", node_zero)

graph.add_edge(START, "router")

# Conditional edge — router calls decide_route to pick next node
graph.add_conditional_edges("router", decide_route)

graph.add_edge("node_positive", END)
graph.add_edge("node_negative", END)
graph.add_edge("node_zero", END)

app = graph.compile()

# Test all three paths
for number in [10, -5, 0]:
    result = app.invoke({"number": number, "result": ""})
    print(result["result"])

Router checking number: 10
10 is positive!
Router checking number: -5
-5 is negative!
Router checking number: 0
That's zero!


In [22]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import TypedDict

class MyState(TypedDict):
    user_input: str
    category: str
    response: str

chat = ChatOllama(model="llama3.1:8b")

# Node 1 — classify the user input
def classify_node(state: MyState) -> MyState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Classify the input into exactly one category.
Reply with only one word, nothing else.
If it is about numbers, calculations, or math → reply: MATH
If it is about countries, cities, capitals, or geography → reply: GEOGRAPHY
Otherwise → reply: OTHER"""),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    result = chain.invoke({"input": state["user_input"]})
    state["category"] = result.strip().upper()
    print(f"Classified as: {state['category']}")
    return state

# Node 2 — handle math questions
def math_node(state: MyState) -> MyState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a math tutor. Give a clear, concise answer."),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    state["response"] = chain.invoke({"input": state["user_input"]})
    return state

# Node 3 — handle geography questions
def geography_node(state: MyState) -> MyState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a geography expert. Give a clear, concise answer."),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    state["response"] = chain.invoke({"input": state["user_input"]})
    return state

# Node 4 — handle everything else
def other_node(state: MyState) -> MyState:
    state["response"] = "I can only answer math and geography questions."
    return state

# Routing function
def route_by_category(state: MyState) -> str:
    if "MATH" in state["category"]:
        return "math_node"
    elif "GEOGRAPHY" in state["category"]:
        return "geography_node"
    else:
        return "other_node"

# Build graph
graph = StateGraph(MyState)
graph.add_node("classify_node", classify_node)
graph.add_node("math_node", math_node)
graph.add_node("geography_node", geography_node)
graph.add_node("other_node", other_node)

graph.add_edge(START, "classify_node")
graph.add_conditional_edges("classify_node", route_by_category)
graph.add_edge("math_node", END)
graph.add_edge("geography_node", END)
graph.add_edge("other_node", END)

app = graph.compile()

# Test all three paths
questions = [
    "What is 15 squared?",
    "What is the capital of Brazil?",
    "What is the meaning of life?"
]

for q in questions:
    print(f"\nQ: {q}")
    result = app.invoke({"user_input": q, "category": "", "response": ""})
    print(f"A: {result['response']}")


Q: What is 15 squared?
A: I can only answer math and geography questions.

Q: What is the capital of Brazil?
A: I can only answer math and geography questions.

Q: What is the meaning of life?
A: I can only answer math and geography questions.


Session 9 — Full Project: Local AI Assistant

In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.tools import tool
from typing import TypedDict, List

# ── State ──────────────────────────────────────────────
class AssistantState(TypedDict):
    user_input: str
    category: str
    response: str
    chat_history: List

# ── Models ─────────────────────────────────────────────
chat = ChatOllama(model="llama3.1:8b")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# ── Memory store ───────────────────────────────────────
memory = ChatMessageHistory()

# ── Vector store ───────────────────────────────────────
vectorstore = FAISS.load_local(
    "./faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("All setup OK")

All setup OK


In [26]:
# ── Tool ───────────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression like '2 + 2' or '10 * 5'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# ── Node 1: Classifier ─────────────────────────────────
def classify_node(state: AssistantState) -> AssistantState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a classifier. Reply with ONE word only. No explanation. No punctuation.
Your only valid responses are: MATH, DOC, or CHAT.
MATH = questions about numbers or calculations
DOC = questions about space, telescopes, planets, moons
CHAT = everything else including greetings and follow-up questions"""),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    result = chain.invoke({"input": state["user_input"]})
    # Take only the first word in case model adds extra text
    state["category"] = result.strip().split()[0].upper()
    print(f"Category: {state['category']}")
    return state
    
# ── Node 2: CALCULATOR ────────────────────────────────────────
def math_node(state: AssistantState) -> AssistantState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Extract the math expression from the question.
Reply with only a valid Python expression using these operators: + - * / **
Use * for multiplication, ** for powers. No words, no symbols like ×."""),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    expression = chain.invoke({"input": state["user_input"]})
    # Clean common unicode math symbols just in case
    expression = expression.strip()
    expression = expression.replace("×", "*").replace("÷", "/").replace("^", "**")
    print(f"Expression: {expression}")
    result = calculator.invoke(expression)
    state["response"] = f"The answer is {result}"
    return state

# ── Node 3: RAG ────────────────────────────────────────
def doc_node(state: AssistantState) -> AssistantState:
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """Answer using only the context below.
If the answer is not in the context, say 'I don't know'.

Context:
{context}"""),
        ("human", "{question}")
    ])
    chain = (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat
        | StrOutputParser()
    )
    state["response"] = chain.invoke(state["user_input"])
    return state

# ── Node 4: Chat with memory ───────────────────────────
def chat_node(state: AssistantState) -> AssistantState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful friendly assistant."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])
    chain = prompt | chat | StrOutputParser()
    response = chain.invoke({
        "input": state["user_input"],
        "history": memory.messages
    })
    # Save to memory
    memory.add_user_message(state["user_input"])
    memory.add_ai_message(response)
    state["response"] = response
    return state

# ── Routing function ───────────────────────────────────
def route(state: AssistantState) -> str:
    if "MATH" in state["category"]:
        return "math_node"
    elif "DOC" in state["category"]:
        return "doc_node"
    else:
        return "chat_node"

print("Nodes defined OK")

Nodes defined OK


In [28]:
# ── Build graph ────────────────────────────────────────
graph = StateGraph(AssistantState)
graph.add_node("classify_node", classify_node)
graph.add_node("math_node", math_node)
graph.add_node("doc_node", doc_node)
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "classify_node")
graph.add_conditional_edges("classify_node", route)
graph.add_edge("math_node", END)
graph.add_edge("doc_node", END)
graph.add_edge("chat_node", END)

app = graph.compile()

# ── Test all four paths ────────────────────────────────
questions = [
    "What is 25 multiplied by 4?",
    "When was the James Webb telescope launched?",
    "What is your name?",
    "What is the full conversation summary?",   # tests memory
]

initial_state = {
    "user_input": "",
    "category": "",
    "response": "",
    "chat_history": []
}

for q in questions:
    print(f"\nQ: {q}")
    initial_state["user_input"] = q
    initial_state["category"] = ""
    initial_state["response"] = ""
    result = app.invoke(initial_state)
    print(f"A: {result['response']}")


Q: What is 25 multiplied by 4?
Category: MATH
Expression: 25 * 4
A: The answer is 100

Q: When was the James Webb telescope launched?
Category: DOC
A: December 25, 2021.

Q: What is your name?
Category: CHAT
A: We've had this conversation already, but I'll repeat myself for fun: My name is Nova, and I'm here to help you with any questions or tasks you'd like assistance with. How can I assist you today?

Q: What is the full conversation summary?
Category: CHAT
A: Here's a quick summary of our conversation so far:

1. You asked me what my name was.
2. I replied that my name is Nova and offered to help answer any questions or provide assistance.
3. You asked me what my name was again (already knowing it was Nova).
4. I repeated myself, reminding you that my name is indeed Nova.
5. You asked me what your own question was from the previous step (and I told you it was "what is my name?").
6. And now, we're here!
